In [1]:
from pathlib import Path

def get_images_from_directory(directory_path):
    image_extensions = {'.jpg', '.jpeg', '.png', '.gif', '.bmp', '.tiff', '.webp'}
    path = Path(directory_path)
    
    images = []
    for file_path in path.iterdir():
        if file_path.is_file() and file_path.suffix.lower() in image_extensions:
            images.append(str(file_path))
    
    return images

# Usage
cat_folder = "/home/dnth/Downloads/archive/PetImages/Cat"
dog_folder = "/home/dnth/Downloads/archive/PetImages/Dog"

cat_image_list = get_images_from_directory(cat_folder)
dog_image_list = get_images_from_directory(dog_folder)

In [2]:
NUM_SAMPLES = 6
image_paths = cat_image_list[:NUM_SAMPLES] + dog_image_list[:NUM_SAMPLES]
image_paths

['/home/dnth/Downloads/archive/PetImages/Cat/12433.jpg',
 '/home/dnth/Downloads/archive/PetImages/Cat/608.jpg',
 '/home/dnth/Downloads/archive/PetImages/Cat/2605.jpg',
 '/home/dnth/Downloads/archive/PetImages/Cat/1683.jpg',
 '/home/dnth/Downloads/archive/PetImages/Cat/4315.jpg',
 '/home/dnth/Downloads/archive/PetImages/Cat/2399.jpg',
 '/home/dnth/Downloads/archive/PetImages/Dog/12433.jpg',
 '/home/dnth/Downloads/archive/PetImages/Dog/608.jpg',
 '/home/dnth/Downloads/archive/PetImages/Dog/2605.jpg',
 '/home/dnth/Downloads/archive/PetImages/Dog/1683.jpg',
 '/home/dnth/Downloads/archive/PetImages/Dog/4315.jpg',
 '/home/dnth/Downloads/archive/PetImages/Dog/2399.jpg']

In [3]:
from setfit import SetFitImageModel, SetFitImageTrainer, TrainingArguments

model = SetFitImageModel(
    timm_model_name="timm/vit_base_patch16_dinov3.lvd1689m",
    train_embeddings=True,
    labels=["cat", "dog"]
)

# Train with image paths and labels
trainer = SetFitImageTrainer(model=model)

# args = TrainingArguments(
#     batch_size=1,
#     num_epochs=4,
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     load_best_model_at_end=True,
# )

trainer.train(x_train=image_paths, y_train=['cat'] * NUM_SAMPLES + ['dog'] * NUM_SAMPLES)
trainer.train(x_train=image_paths, y_train=['cat'] * NUM_SAMPLES + ['dog'] * NUM_SAMPLES)
trainer.train(x_train=image_paths, y_train=['cat'] * NUM_SAMPLES + ['dog'] * NUM_SAMPLES)



# Predict on new images
# prediction = model.predict("/home/dnth/Downloads/archive/PetImages/Cat/0.jpg")
# probabilities = model.predict_proba("/home/dnth/Downloads/archive/PetImages/Cat/0.jpg")


Training TIMM model embeddings for image model


Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Iteration:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 1/1, Loss: 2.2792
Training TIMM model embeddings for image model


Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Iteration:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 1/1, Loss: 2.0532
Training TIMM model embeddings for image model


Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Iteration:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 1/1, Loss: 2.1270


In [4]:
model.predict("/home/dnth/Downloads/archive/PetImages/Cat/0.jpg", show_progress_bar=True)

Encoding images:   0%|          | 0/1 [00:00<?, ?batch/s]

'cat'

In [5]:
model.predict_proba("/home/dnth/Downloads/archive/PetImages/Cat/0.jpg", show_progress_bar=True)


Encoding images:   0%|          | 0/1 [00:00<?, ?batch/s]

tensor([0.7239, 0.2761], dtype=torch.float64)

## Eval on Cat Dogs dataset

In [6]:
import pandas as pd

image_list = cat_image_list + dog_image_list

df = pd.DataFrame(image_list, columns=['image_path'])
df


,image_path
0,/home/dnth/Downloads/archive/PetImages/Cat/124...
1,/home/dnth/Downloads/archive/PetImages/Cat/608...
2,/home/dnth/Downloads/archive/PetImages/Cat/260...
3,/home/dnth/Downloads/archive/PetImages/Cat/168...
4,/home/dnth/Downloads/archive/PetImages/Cat/431...
...,...
24993,/home/dnth/Downloads/archive/PetImages/Dog/978...
24994,/home/dnth/Downloads/archive/PetImages/Dog/600...
24995,/home/dnth/Downloads/archive/PetImages/Dog/559...
24996,/home/dnth/Downloads/archive/PetImages/Dog/951...


In [7]:
df['label'] = df['image_path'].str.split('/').str[-2].str.lower()

In [8]:
df

,image_path,label
0,/home/dnth/Downloads/archive/PetImages/Cat/124...,cat
1,/home/dnth/Downloads/archive/PetImages/Cat/608...,cat
2,/home/dnth/Downloads/archive/PetImages/Cat/260...,cat
3,/home/dnth/Downloads/archive/PetImages/Cat/168...,cat
4,/home/dnth/Downloads/archive/PetImages/Cat/431...,cat
...,...,...
24993,/home/dnth/Downloads/archive/PetImages/Dog/978...,dog
24994,/home/dnth/Downloads/archive/PetImages/Dog/600...,dog
24995,/home/dnth/Downloads/archive/PetImages/Dog/559...,dog
24996,/home/dnth/Downloads/archive/PetImages/Dog/951...,dog


In [9]:
df = df.sample(1000)

In [10]:
# Run batch inference
df['pred'] = model.predict(df['image_path'].tolist(), batch_size=64, show_progress_bar=True)

Encoding images:   0%|          | 0/16 [00:00<?, ?batch/s]

In [11]:
df

,image_path,label,pred
18164,/home/dnth/Downloads/archive/PetImages/Dog/306...,dog,dog
7890,/home/dnth/Downloads/archive/PetImages/Cat/975...,cat,cat
22961,/home/dnth/Downloads/archive/PetImages/Dog/240...,dog,dog
2981,/home/dnth/Downloads/archive/PetImages/Cat/603...,cat,cat
7024,/home/dnth/Downloads/archive/PetImages/Cat/334...,cat,cat
...,...,...,...
1918,/home/dnth/Downloads/archive/PetImages/Cat/101...,cat,cat
17389,/home/dnth/Downloads/archive/PetImages/Dog/617...,dog,dog
8205,/home/dnth/Downloads/archive/PetImages/Cat/438...,cat,cat
17497,/home/dnth/Downloads/archive/PetImages/Dog/341...,dog,dog


In [12]:
accuracy = (df['label'] == df['pred']).mean()
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.9860


In [13]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Accuracy (same as above)
acc = accuracy_score(df['label'], df['pred'])
print(f"Accuracy: {acc:.4f}\n")

# Detailed classification report
print("Classification Report:")
print(classification_report(df['label'], df['pred']))

# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(df['label'], df['pred']))

Accuracy: 0.9860

Classification Report:
              precision    recall  f1-score   support

         cat       0.98      0.99      0.99       514
         dog       0.99      0.98      0.99       486

    accuracy                           0.99      1000
   macro avg       0.99      0.99      0.99      1000
weighted avg       0.99      0.99      0.99      1000

Confusion Matrix:
[[511   3]
 [ 11 475]]
